# NB_bishop_ch03_figures

**Bishop Chapter 3 — Key Figures: Bernoulli, Binomial, Beta, Gaussian, Student-t, Mixture Models & Bayesian Updating**

This notebook generates pedagogical matplotlib figures for Bishop Chapter 3 on standard distributions.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_03/NB_bishop_ch03_figures.ipynb)

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import gamma as gamma_fn
import os

# ── colour palette ──────────────────────────────────────────────
COLORS = {
    'blue':  '#1A3A6E',
    'red':   '#CD0000',
    'green': '#2E7D32',
    'amber': '#B5853F',
}

# ── global rcParams ─────────────────────────────────────────────
matplotlib.rcParams['figure.facecolor']   = 'none'
matplotlib.rcParams['axes.facecolor']     = 'none'
matplotlib.rcParams['savefig.facecolor']  = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid']          = False
matplotlib.rcParams['legend.loc']         = 'upper center'
matplotlib.rcParams['legend.framealpha']  = 0.0

CHART_DIR = '../../charts'
os.makedirs(CHART_DIR, exist_ok=True)

def save_fig(fig, name, chart_dir=CHART_DIR):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

print('Setup complete.')

## Figure 3.1 — Bernoulli & Binomial Distributions

In [ ]:
p = 0.3

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Panel (a): Bernoulli PMF
ax = axes[0]
x_bern = [0, 1]
pmf_bern = [1 - p, p]
ax.bar(x_bern, pmf_bern, width=0.4, color=[COLORS['blue'], COLORS['red']],
       edgecolor='k', linewidth=0.8)
ax.set_xticks([0, 1])
ax.set_xticklabels(['$x=0$', '$x=1$'])
ax.set_ylabel('$p(x)$')
ax.set_title(f'Bernoulli ($\\mu={p}$)')
ax.set_ylim(0, 1.0)

# Panels (b)-(d): Binomial PMFs for N=5, 10, 20
Ns = [5, 10, 20]
cols = [COLORS['blue'], COLORS['red'], COLORS['green']]

for ax, N, c in zip(axes[1:], Ns, cols):
    k = np.arange(0, N + 1)
    pmf = stats.binom.pmf(k, N, p)
    ax.bar(k, pmf, width=0.6, color=c, edgecolor='k', linewidth=0.5, alpha=0.85)
    ax.set_xlabel('$k$')
    ax.set_ylabel('$p(k \\mid N, \\mu)$')
    ax.set_title(f'Binomial ($N={N},\\; \\mu={p}$)')
    # Mark the mean
    mean_val = N * p
    ax.axvline(mean_val, ls='--', color='k', lw=1, alpha=0.5)
    ax.text(mean_val + 0.3, max(pmf) * 0.9, f'$\\mathbb{{E}}[k]={mean_val:.0f}$',
            fontsize=8, va='top')

fig.suptitle('Bernoulli and Binomial Distributions', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, 'fig3_1_bernoulli_binomial')
plt.show()

## Figure 3.4 — Beta Distribution (Conjugate Prior for Bernoulli)

In [ ]:
x = np.linspace(0.005, 0.995, 500)

ab_pairs = [(0.5, 0.5), (1, 1), (2, 5), (5, 2), (2, 2)]
colors   = [COLORS['red'], 'gray', COLORS['blue'], COLORS['green'], COLORS['amber']]
styles   = ['-', '--', '-', '-', '-.']

fig, ax = plt.subplots(figsize=(8, 5))

for (a, b), c, ls in zip(ab_pairs, colors, styles):
    y = stats.beta.pdf(x, a, b)
    ax.plot(x, y, color=c, lw=2.5, ls=ls, label=f'$a={a},\\; b={b}$')

ax.set_xlabel('$\\mu$')
ax.set_ylabel('$\\mathrm{Beta}(\\mu \\mid a, b)$')
ax.set_title('Beta Distribution — Conjugate Prior for Bernoulli/Binomial')
ax.set_xlim(0, 1)
ax.set_ylim(0, 4.0)
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig3_4_beta_distribution')
plt.show()

## Figure 3.5 — Univariate and Bivariate Gaussian

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Panel (a): 1D Gaussian with mean/std shaded ─────────────
ax = axes[0]
mu, sigma = 0, 1
x = np.linspace(-4, 4, 500)
y = stats.norm.pdf(x, mu, sigma)

ax.plot(x, y, color=COLORS['blue'], lw=2.5)
ax.fill_between(x, y, where=(x >= mu - sigma) & (x <= mu + sigma),
                alpha=0.25, color=COLORS['blue'], label='$\\mu \\pm \\sigma$ (68.3%)')
ax.fill_between(x, y, where=(x >= mu - 2*sigma) & (x <= mu + 2*sigma) &
                ~((x >= mu - sigma) & (x <= mu + sigma)),
                alpha=0.12, color=COLORS['amber'], label='$\\mu \\pm 2\\sigma$ (95.4%)')

ax.axvline(mu, ls='--', color=COLORS['red'], lw=1.5, alpha=0.7)
ax.annotate('$\\mu$', xy=(mu, max(y)), xytext=(mu + 0.3, max(y) + 0.02),
            fontsize=13, color=COLORS['red'])

# sigma arrows
y_arrow = 0.24
ax.annotate('', xy=(mu + sigma, y_arrow), xytext=(mu, y_arrow),
            arrowprops=dict(arrowstyle='<->', color=COLORS['green'], lw=1.5))
ax.text(mu + sigma/2, y_arrow + 0.015, '$\\sigma$', ha='center', fontsize=12,
        color=COLORS['green'])

ax.set_xlabel('$x$')
ax.set_ylabel('$\\mathcal{N}(x \\mid \\mu, \\sigma^2)$')
ax.set_title('Univariate Gaussian')
ax.legend()

# ── Panel (b): 2D Gaussian contours ─────────────────────────
ax = axes[1]

# Three different covariance structures
configs = [
    (np.array([0, 0]),   np.array([[1, 0], [0, 1]]),       COLORS['blue'],  'Spherical'),
    (np.array([0, 0]),   np.array([[2, 0.8], [0.8, 0.5]]), COLORS['red'],   'Full'),
    (np.array([0, 0]),   np.array([[0.5, 0], [0, 2]]),     COLORS['green'], 'Diagonal'),
]

grid = np.linspace(-4, 4, 200)
X, Y = np.meshgrid(grid, grid)
pos = np.dstack((X, Y))

for mu, cov, c, label in configs:
    rv = stats.multivariate_normal(mu, cov)
    Z = rv.pdf(pos)
    ax.contour(X, Y, Z, levels=4, colors=c, linewidths=1.8, alpha=0.85)
    # Invisible plot for legend
    ax.plot([], [], color=c, lw=2, label=label)

ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Bivariate Gaussian — Covariance Structures')
ax.set_aspect('equal')
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig3_5_gaussian_1d_2d')
plt.show()

## Figure 3.8 — Student-t Distribution (Heavy Tails)

In [ ]:
x = np.linspace(-6, 6, 600)

nus = [1, 3, 10]
colors_t = [COLORS['red'], COLORS['amber'], COLORS['green']]
labels_t = ['$\\nu=1$ (Cauchy)', '$\\nu=3$', '$\\nu=10$']

fig, ax = plt.subplots(figsize=(8, 5))

# Gaussian reference
ax.plot(x, stats.norm.pdf(x), color=COLORS['blue'], lw=2.5, label='$\\nu=\\infty$ (Gaussian)')

for nu, c, lab in zip(nus, colors_t, labels_t):
    ax.plot(x, stats.t.pdf(x, nu), color=c, lw=2, label=lab)

# Highlight heavy tails
ax.fill_between(x, stats.t.pdf(x, 1), stats.norm.pdf(x),
                where=np.abs(x) > 2.5, alpha=0.15, color=COLORS['red'])
ax.annotate('Heavy tails', xy=(4.0, 0.02), fontsize=10,
            color=COLORS['red'], ha='center', style='italic')

ax.set_xlabel('$x$')
ax.set_ylabel('$p(x)$')
ax.set_title("Student-t Distribution — Effect of Degrees of Freedom $\\nu$")
ax.set_xlim(-6, 6)
ax.set_ylim(0, 0.42)
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig3_8_student_t')
plt.show()

## Figure 3.9 — Gaussian Mixture Model

In [ ]:
x = np.linspace(-6, 10, 600)

# Mixture components: (weight, mean, std)
components = [
    (0.35, -1.0, 0.8),
    (0.45,  3.0, 1.2),
    (0.20,  6.5, 0.6),
]
comp_colors = [COLORS['blue'], COLORS['red'], COLORS['green']]
comp_labels = ['Component 1', 'Component 2', 'Component 3']

fig, ax = plt.subplots(figsize=(9, 5))

mixture = np.zeros_like(x)

for (w, mu, sigma), c, lab in zip(components, comp_colors, comp_labels):
    y = w * stats.norm.pdf(x, mu, sigma)
    mixture += y
    ax.plot(x, y, color=c, lw=1.8, ls='--', alpha=0.7, label=f'{lab}: $\\pi={w}$, $\\mu={mu}$')
    ax.fill_between(x, y, alpha=0.06, color=c)

ax.plot(x, mixture, color='k', lw=3, label='Mixture $p(x)$')

ax.set_xlabel('$x$')
ax.set_ylabel('$p(x)$')
ax.set_title('Gaussian Mixture Model ($K=3$)')
ax.set_xlim(-5, 9.5)
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig3_9_mixture_gaussians')
plt.show()

## Figure 3.6 — Conditional Distribution of a 2D Gaussian

In [ ]:
# 2D Gaussian with conditional slice
mu = np.array([1.0, 0.5])
cov = np.array([[1.0, 0.7],
                [0.7, 1.0]])

grid = np.linspace(-3, 5, 300)
X, Y = np.meshgrid(grid, grid)
pos = np.dstack((X, Y))
rv = stats.multivariate_normal(mu, cov)
Z = rv.pdf(pos)

# Conditional: p(x2 | x1 = x1_val)
x1_val = 2.5
cond_mu = mu[1] + cov[1, 0] / cov[0, 0] * (x1_val - mu[0])
cond_var = cov[1, 1] - cov[1, 0]**2 / cov[0, 0]
cond_std = np.sqrt(cond_var)

fig, ax = plt.subplots(figsize=(8, 7))

# Contours of joint distribution
cs = ax.contourf(X, Y, Z, levels=15, cmap='Blues', alpha=0.4)
ax.contour(X, Y, Z, levels=8, colors=COLORS['blue'], linewidths=0.8, alpha=0.6)

# Vertical slice line
ax.axvline(x1_val, color=COLORS['red'], lw=2, ls='--', alpha=0.8,
           label=f'$x_1 = {x1_val}$')

# Draw conditional distribution as a curve along the slice
y_cond = np.linspace(-3, 5, 300)
p_cond = stats.norm.pdf(y_cond, cond_mu, cond_std)
# Scale and shift to overlay on the plot
p_scaled = p_cond / p_cond.max() * 1.2
ax.plot(x1_val + p_scaled, y_cond, color=COLORS['red'], lw=2.5,
        label=f'$p(x_2 \\mid x_1={x1_val})$')
ax.fill_betweenx(y_cond, x1_val, x1_val + p_scaled, alpha=0.15, color=COLORS['red'])

# Mark conditional mean
ax.plot(x1_val, cond_mu, 'o', ms=8, color=COLORS['red'], zorder=5,
        markeredgecolor='k', markeredgewidth=0.8)
ax.annotate(f'$\\mathbb{{E}}[x_2 \\mid x_1] = {cond_mu:.2f}$',
            xy=(x1_val, cond_mu), xytext=(x1_val + 1.8, cond_mu + 0.8),
            fontsize=11, color=COLORS['red'],
            arrowprops=dict(arrowstyle='->', color=COLORS['red'], lw=1.2))

# Mark joint mean
ax.plot(*mu, 's', ms=8, color=COLORS['green'], zorder=5,
        markeredgecolor='k', markeredgewidth=0.8, label='Joint mean')

ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Conditional Distribution from a Bivariate Gaussian')
ax.set_xlim(-2.5, 5)
ax.set_ylim(-2.5, 4)
ax.set_aspect('equal')
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig3_6_gaussian_conditional')
plt.show()

## Figure 3.12 — Sequential Bayesian Update (Conjugate Prior)

In [ ]:
np.random.seed(42)

# True parameter
mu_true = 0.6

# Generate observations from Bernoulli(mu_true)
N_total = 20
observations = (np.random.rand(N_total) < mu_true).astype(int)

# Prior: Beta(a0, b0)
a0, b0 = 2, 2

# Stages to show: prior, after 1, after 5, after 20 observations
stages = [0, 1, 5, 20]
stage_labels = ['Prior', 'After $N=1$', 'After $N=5$', 'After $N=20$']
stage_colors = ['gray', COLORS['amber'], COLORS['green'], COLORS['blue']]

mu_grid = np.linspace(0.001, 0.999, 500)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, n, lab, c in zip(axes, stages, stage_labels, stage_colors):
    # Posterior after n observations: Beta(a0 + sum(x), b0 + n - sum(x))
    if n == 0:
        a_post, b_post = a0, b0
    else:
        heads = observations[:n].sum()
        a_post = a0 + heads
        b_post = b0 + n - heads

    posterior = stats.beta.pdf(mu_grid, a_post, b_post)
    ax.plot(mu_grid, posterior, color=c, lw=2.5)
    ax.fill_between(mu_grid, posterior, alpha=0.15, color=c)

    # Mark true value
    ax.axvline(mu_true, ls='--', color=COLORS['red'], lw=1.5, alpha=0.7)

    # Mark MAP estimate (mode of Beta)
    if a_post > 1 and b_post > 1:
        mode = (a_post - 1) / (a_post + b_post - 2)
        ax.axvline(mode, ls=':', color='k', lw=1, alpha=0.5)
        ax.text(mode + 0.03, max(posterior) * 0.85,
                f'MAP={mode:.2f}', fontsize=8, color='k')

    ax.set_xlabel('$\\mu$')
    ax.set_ylabel('$p(\\mu \\mid \\mathcal{D})$')
    ax.set_title(f'{lab}\n$\\mathrm{{Beta}}({a_post}, {b_post})$')
    ax.set_xlim(0, 1)

    # Show observations as rug plot
    if n > 0:
        obs_n = observations[:n]
        for xi in obs_n:
            ax.plot(xi, -0.1 * max(posterior), '|', ms=8, color=COLORS['red'],
                    markeredgewidth=1.5, clip_on=False)

# Add true value annotation to last panel
axes[-1].annotate(f'True $\\mu={mu_true}$', xy=(mu_true, 0),
                  xytext=(mu_true - 0.25, max(posterior) * 0.4),
                  fontsize=9, color=COLORS['red'],
                  arrowprops=dict(arrowstyle='->', color=COLORS['red'], lw=1))

fig.suptitle('Sequential Bayesian Updating: Beta–Bernoulli Conjugate Model',
             fontsize=14, y=1.04)
fig.tight_layout()
save_fig(fig, 'fig3_12_conjugate_update')
plt.show()